# Assignment 2.5 — Stable Diffusion LoRA

Simple version for **Kaggle Tesla P100 16GB**.

Flow:
1. Load 5–10 personal images
2. Add LoRA to SD 1.5 U-Net
3. Train LoRA
4. Generate test images
5. Compute CLIP-I / CLIP-T
6. Save LoRA for ComfyUI

## 1. Kaggle P100 setup

Run this cell **before importing torch**.

Kaggle currently may ship a PyTorch build without P100 (`sm_60`) support.
`torchao` is not needed for this assignment, so we remove it to avoid the PEFT version conflict.

In [ ]:
import subprocess, sys, torch

# Only Pascal P100 needs Kaggle's compatibility wheel. Keep the installed
# PyTorch on T4/newer GPUs to avoid disrupting the live kernel.
capability = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
is_p100 = capability == (6, 0)
needs_p100_torch = is_p100 and "sm_60" not in torch.cuda.get_arch_list()
print("GPU capability:", capability)

if needs_p100_torch:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "torch==2.7.0", "torchvision==0.22.0", "torchaudio==2.7.0",
        "--index-url", "https://download.pytorch.org/whl/cu126",
    ])
else:
    print("Keeping Kaggle's existing PyTorch wheel for this GPU.")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.35.1", "transformers==4.53.3", "accelerate==1.9.0",
    "peft==0.17.1", "safetensors",
])

## 2. Check P100

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Capability:', torch.cuda.get_device_capability(0))
print('Supported arch:', torch.cuda.get_arch_list())
print('Any CUDA GPU is supported; P100 and T4 are both valid for this assignment.')

## 3. Find your 5–10 personal images

Add your private image dataset to the Kaggle notebook.

This cell searches all images under `/kaggle/input`, so you do not need to guess the dataset folder name.

In [ ]:
from pathlib import Path
import zipfile

# Nếu input là file zip -> giải nén
zip_files = list(Path("/kaggle/input").rglob("*.zip"))

for z in zip_files:
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/images")

# Tìm ảnh cả ở input lẫn folder vừa giải nén
image_paths = [
    p for root in [Path("/kaggle/input"), Path("/kaggle/working/images")]
    if root.exists()
    for p in root.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
]

print("Number of images:", len(image_paths))
for p in image_paths:
    print(p)

## 4. Load Stable Diffusion 1.5 and add LoRA

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as T

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig

device = "cuda"
dtype = torch.float16
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

tokenizer = CLIPTokenizer.from_pretrained(
    model_id,
    subfolder="tokenizer"
)

text_encoder = CLIPTextModel.from_pretrained(
    model_id,
    subfolder="text_encoder",
    torch_dtype=dtype
).to(device)

vae = AutoencoderKL.from_pretrained(
    model_id,
    subfolder="vae",
    torch_dtype=dtype
).to(device)

unet = UNet2DConditionModel.from_pretrained(
    model_id,
    subfolder="unet",
    torch_dtype=dtype
).to(device)

noise_scheduler = DDPMScheduler.from_pretrained(
    model_id,
    subfolder="scheduler"
)

# Freeze base model
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

# Add LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
)

unet.add_adapter(lora_config)

# QUAN TRỌNG:
# Base U-Net vẫn FP16, nhưng LoRA trainable params phải FP32
for p in unet.parameters():
    if p.requires_grad:
        p.data = p.data.float()

trainable_params = [
    p for p in unet.parameters()
    if p.requires_grad
]

print("Number trainable params:", sum(p.numel() for p in trainable_params))
print("LoRA dtype:", trainable_params[0].dtype)

## 5. Dataset

Change the prompt to match your subject.

Examples:
- person: `a photo of sks person`
- dog: `a photo of sks dog`
- cat: `a photo of sks cat`

In [ ]:
instance_prompt = "a photo of sks person"

transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

class PersonalDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        image = Image.open(self.paths[i]).convert("RGB")
        return transform(image)

dataset = PersonalDataset(image_paths)
loader = DataLoader(dataset, batch_size=1, shuffle=True)

## 6. Encode the training prompt once

In [ ]:
tokens = tokenizer(
    instance_prompt,
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt",
).input_ids.to(device)

with torch.no_grad():
    text_emb = text_encoder(tokens)[0]

## 7. Train LoRA

Only LoRA parameters are updated.

In [ ]:
trainable_params = [
    p for p in unet.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=1e-4
)

max_steps = 300
step = 0

unet.train()

while step < max_steps:

    for images in loader:

        images = images.to(
            device,
            dtype=dtype
        )

        # image -> latent
        with torch.no_grad():

            latents = vae.encode(
                images
            ).latent_dist.sample()

            latents = (
                latents *
                vae.config.scaling_factor
            )

        # noise
        noise = torch.randn_like(latents)

        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],),
            device=device
        ).long()

        noisy_latents = noise_scheduler.add_noise(
            latents,
            noise,
            timesteps
        )

        # predict noise
        noise_pred = unet(
            noisy_latents,
            timesteps,
            encoder_hidden_states=text_emb
        ).sample

        # calculate loss in FP32
        loss = F.mse_loss(
            noise_pred.float(),
            noise.float()
        )

        optimizer.zero_grad()

        loss.backward()

        # tránh gradient explode
        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            1.0
        )

        optimizer.step()

        step += 1

        if step % 20 == 0:
            print(
                f"step {step}: "
                f"loss = {loss.item():.4f}"
            )

        if step >= max_steps:
            break

## 8. Save LoRA

In [ ]:
# output_dir = "/kaggle/working/my_lora"
# unet.save_pretrained(output_dir)

# print("Saved LoRA to:", output_dir)

In [ ]:
from pathlib import Path
from peft.utils import get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_kohya
from safetensors.torch import save_file

lora_dir = Path('/kaggle/working/my_lora')
lora_dir.mkdir(exist_ok=True)

# PEFT state from a bare UNet has no "unet." prefix. Add it before
# Diffusers converts to the Kohya/ComfyUI lora_unet_* key convention.
peft_state = get_peft_model_state_dict(unet)
peft_state = {
    key if key.startswith('unet.') else f'unet.{key}': value
    for key, value in peft_state.items()
}
kohya_state = convert_state_dict_to_kohya(peft_state)
comfy_lora_path = lora_dir / 'my_lora_comfy.safetensors'
save_file(kohya_state, str(comfy_lora_path))

print('Saved ComfyUI LoRA:', comfy_lora_path)
print('Example key:', next(iter(kohya_state)))
print('LoRA tensors:', len(kohya_state))

## 9. Generate test images

In [ ]:
import numpy as np
from diffusers import DDIMScheduler

scheduler = DDIMScheduler.from_pretrained(
    model_id, subfolder="scheduler"
)

def encode_prompt(text):
    ids = tokenizer(
        text,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    ).input_ids.to(device)

    with torch.no_grad():
        return text_encoder(ids)[0]

@torch.no_grad()
def generate(prompt, seed=42, scale=7.5, steps=30):
    cond = encode_prompt(prompt)
    uncond = encode_prompt("")

    generator = torch.Generator(device=device).manual_seed(seed)

    latents = torch.randn(
        (1, unet.config.in_channels, 64, 64),
        generator=generator,
        device=device,
        dtype=dtype,
    )

    scheduler.set_timesteps(steps, device=device)

    for t in scheduler.timesteps:
        noise_u = unet(latents, t, encoder_hidden_states=uncond).sample
        noise_c = unet(latents, t, encoder_hidden_states=cond).sample

        noise = noise_u + scale * (noise_c - noise_u)

        latents = scheduler.step(
            noise, t, latents
        ).prev_sample

    image = vae.decode(
        latents / vae.config.scaling_factor
    ).sample

    image = (image / 2 + 0.5).clamp(0, 1)
    image = image[0].cpu().permute(1, 2, 0).numpy()
    image = (image * 255).astype(np.uint8)

    return Image.fromarray(image)

## 10. Generate evaluation images

In [ ]:
eval_prompts = [
    instance_prompt,
    instance_prompt + " wearing sunglasses",
    instance_prompt + " in a park",
]

generated_images = [
    generate(prompt, seed=42 + i)
    for i, prompt in enumerate(eval_prompts)
]

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, image, prompt in zip(axes, generated_images, eval_prompts):
    ax.imshow(image)
    ax.set_title(prompt)
    ax.axis("off")

plt.show()

## 11. CLIP-I and CLIP-T

- **CLIP-I:** generated image vs reference subject images
- **CLIP-T:** generated image vs text prompt

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

reference_images = [
    Image.open(p).convert("RGB")
    for p in image_paths
]

@torch.no_grad()
def image_features(images):
    inputs = clip_processor(
        images=images,
        return_tensors="pt"
    ).to(device)

    features = clip_model.get_image_features(**inputs)
    return F.normalize(features, dim=-1)

@torch.no_grad()
def text_features(texts):
    inputs = clip_processor(
        text=texts,
        padding=True,
        return_tensors="pt"
    ).to(device)

    features = clip_model.get_text_features(**inputs)
    return F.normalize(features, dim=-1)

ref = image_features(reference_images).mean(dim=0, keepdim=True)
ref = F.normalize(ref, dim=-1)

gen = image_features(generated_images)
txt = text_features(eval_prompts)

clip_i = (gen @ ref.T).squeeze(1)
clip_t = (gen * txt).sum(dim=1)

for prompt, score_i, score_t in zip(
    eval_prompts, clip_i.tolist(), clip_t.tolist()
):
    print(prompt)
    print("CLIP-I:", round(score_i, 4))
    print("CLIP-T:", round(score_t, 4))
    print()

## 12. ComfyUI workflow

The cell below saves `assignment_2_5_comfy_api.json` in Output. It connects SD 1.5, the trained `my_lora_comfy.safetensors`, text prompts, KSampler and Save Image. Copy the checkpoint and LoRA to ComfyUI before inference.

In [ ]:
import json
from pathlib import Path

# API-format ComfyUI workflow; copy the trained LoRA into ComfyUI/models/loras.
workflow = {
    '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'v1-5-pruned-emaonly.safetensors'}},
    '2': {'class_type': 'LoraLoader', 'inputs': {'model': ['1', 0], 'clip': ['1', 1], 'lora_name': 'my_lora_comfy.safetensors', 'strength_model': 1.0, 'strength_clip': 1.0}},
    '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': instance_prompt, 'clip': ['2', 1]}},
    '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': '', 'clip': ['2', 1]}},
    '5': {'class_type': 'EmptyLatentImage', 'inputs': {'width': 512, 'height': 512, 'batch_size': 1}},
    '6': {'class_type': 'KSampler', 'inputs': {'seed': 42, 'steps': 20, 'cfg': 7.5, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0, 'model': ['2', 0], 'positive': ['3', 0], 'negative': ['4', 0], 'latent_image': ['5', 0]}},
    '7': {'class_type': 'VAEDecode', 'inputs': {'samples': ['6', 0], 'vae': ['1', 2]}},
    '8': {'class_type': 'SaveImage', 'inputs': {'filename_prefix': 'assignment_2_5', 'images': ['7', 0]}},
}
out = Path('/kaggle/working/assignment_2_5_comfy_api.json')
out.write_text(json.dumps(workflow, indent=2))
print('Saved:', out)


In [ ]:
# Workflow JSON saved in the cell above.